# Hypothesis results: sandwich profitability on AMMs

Notebook for thesis-facing charts from simulator CSV outputs. Expected inputs:

- `../results/sweep.csv` or `../results/output.csv` from `mev-sim`
- `../results/zhou_vs_numerical.csv` from `bench_zhou_vs_numerical`
- `../results/real_pool_comparison.csv` from `compare_real_pool`


## A. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
ROOT = Path("..").resolve()
RESULTS = ROOT / "results"

def first_existing(*names):
    for name in names:
        path = RESULTS / name
        if path.exists():
            return path
    return None

main_path = first_existing("sweep.csv", "output.csv")
zhou_path = first_existing("zhou_vs_numerical.csv")
real_cmp_path = first_existing("real_pool_comparison.csv")

main = pd.read_csv(main_path) if main_path else pd.DataFrame()
zhou = pd.read_csv(zhou_path) if zhou_path else pd.DataFrame()
real_cmp = pd.read_csv(real_cmp_path) if real_cmp_path else pd.DataFrame()

print("main:", main_path, main.shape)
print("zhou:", zhou_path, zhou.shape)
print("real_cmp:", real_cmp_path, real_cmp.shape)


## B. Main simulator summary

In [ ]:
if main.empty:
    print("No main simulator CSV found. Run: cargo run --bin mev-sim -- -c configs/sweep_liquidity.toml -o results/sweep.csv --parallel")
else:
    summary = (
        main.groupby(["source", "strategy", "attack_status"], dropna=False)
        .agg(
            rows=("attack_status", "size"),
            profitable_rate=("attack_profitable", "mean"),
            feasible_rate=("attack_feasible", "mean"),
            median_net_profit=("attacker_net_profit", "median"),
            median_victim_loss_bps=("victim_loss_bps_of_fair_out", "median"),
        )
        .reset_index()
    )
    display(summary)


## C. Profitability heatmap

In [ ]:
if {"pool_fee_bps", "victim_size_bps_of_reserve", "attack_feasible"}.issubset(main.columns):
    heat = main.pivot_table(
        index="victim_size_bps_of_reserve",
        columns="pool_fee_bps",
        values="attack_feasible",
        aggfunc="mean",
    ).sort_index()
    plt.figure(figsize=(9, 5))
    sns.heatmap(heat, annot=True, fmt=".2f", cmap="viridis", cbar_kws={"label": "feasible rate"})
    plt.title("Attack feasibility by victim size and fee tier")
    plt.xlabel("fee tier [bps]")
    plt.ylabel("victim size [bps of reserve_in]")
    plt.tight_layout()
else:
    print("Missing columns for feasibility heatmap")


## D. Fee tier and tx cost sensitivity

In [ ]:
if {"pool_fee_bps", "attacker_net_profit", "tx_cost_total"}.issubset(main.columns):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fee_line = main.groupby("pool_fee_bps")["attacker_net_profit"].median().reset_index()
    sns.lineplot(data=fee_line, x="pool_fee_bps", y="attacker_net_profit", marker="o", ax=axes[0])
    axes[0].axhline(0, color="black", linewidth=1)
    axes[0].set_title("Median net profit by fee tier")
    axes[0].set_xlabel("fee tier [bps]")
    axes[0].set_ylabel("median net profit [token_in units]")

    cost_line = main.groupby("tx_cost_total")["attack_profitable"].mean().reset_index()
    sns.lineplot(data=cost_line, x="tx_cost_total", y="attack_profitable", marker="o", ax=axes[1])
    axes[1].set_title("Profitable rate by total tx cost")
    axes[1].set_xlabel("tx_cost_total [token_in units]")
    axes[1].set_ylabel("profitable rate")
    plt.tight_layout()
else:
    print("Missing columns for fee/tx-cost plots")


## E. Zhou closed-form vs numerical optimizer

In [ ]:
if zhou.empty:
    print("No Zhou benchmark CSV found. Run: cargo run -p simulator --example bench_zhou_vs_numerical -- -o results/zhou_vs_numerical.csv")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    sns.lineplot(data=zhou, x="fee_bps", y="profit_gap_abs", hue="liquidity_depth_label", estimator="median", errorbar=None, ax=axes[0])
    axes[0].axhline(0, color="black", linewidth=1)
    axes[0].set_title("Median numerical - closed-form profit gap")
    axes[0].set_ylabel("profit gap [token_in units]")

    gap_heat = zhou.pivot_table(index="victim_bps_of_reserve", columns="fee_bps", values="profit_gap_rel", aggfunc="median")
    sns.heatmap(gap_heat, cmap="mako", center=0, ax=axes[1], cbar_kws={"label": "relative gap"})
    axes[1].set_title("Relative profit gap heatmap")
    axes[1].set_xlabel("fee tier [bps]")
    axes[1].set_ylabel("victim size [bps of reserve]")
    plt.tight_layout()


## F. Synthetic vs Raydium snapshot comparison

In [ ]:
if real_cmp.empty:
    print("No real-pool comparison CSV found. Run: cargo run -p simulator --bin compare_real_pool -- -c configs/sweep_real_pool.toml -o results/real_pool_comparison.csv")
else:
    cmp_summary = (
        real_cmp.groupby(["source", "pool_label"], dropna=False)
        .agg(
            rows=("source", "size"),
            profitable_rate=("attack_profitable", "mean"),
            feasible_rate=("attack_feasible", "mean"),
            median_net_profit=("attacker_net_profit", "median"),
            median_victim_loss_bps=("victim_loss_bps_of_fair_out", "median"),
        )
        .reset_index()
    )
    display(cmp_summary)

    plt.figure(figsize=(8, 4))
    sns.barplot(data=real_cmp, x="source", y="attack_feasible", errorbar=None)
    plt.title("Feasible attack rate: synthetic matched state vs Raydium snapshot")
    plt.xlabel("")
    plt.ylabel("feasible rate")
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
